# QPEI Validation Round 3
## Ordinal Measurement → Parallel Analysis → Multi-Solution EFA → Framework Crosswalk → CFA → Decision Log → QPEI Construction Path

This notebook is the methodological sequel to Round 2.

### What Round 3 changes

1. Uses the **corrected master Excel path**.
2. Preserves the established **35 QPEI indicators / 6-domain framework**.
3. Correctly handles **1–5 ordinal responses** and **88/99 missing codes**.
4. Uses a properly constructed **ordinal correlation workflow** (polychoric with documented Spearman fallback).
5. Checks **matrix positive-definiteness** before EFA and applies a documented ridge repair if needed.
6. Computes **factorability** (KMO + Bartlett) on the ordinal matrix.
7. Runs **parallel analysis + scree + MAP** evidence.
8. Compares **plausible EFA solutions** (k−1, k, k+1) rather than blindly accepting one.
9. Flags weak loadings, cross-loadings, and low communalities with explicit thresholds.
10. Produces the **empirical factor ↔ QPEI framework crosswalk** as a *review template* (no automatic Factor = Domain assignment).
11. Separates **reflective survey-scale evidence** from the **formative QPEI construction**.
12. Runs **candidate CFA models only after** the EFA evidence and crosswalk review flags.
13. Produces tables/figures in Colab and saves them to the Round-3 output folder.
14. Produces a comprehensive **`results_round3.json`**.
15. Produces a final **machine-readable decision log** so every item’s retain / review / drop rationale is explicit.

### Critical design rule

The code **never** automatically decides:

> “Factor 1 = D1, Factor 2 = D2 …”

Empirical factors are labelled only as F1, F2, …  
Mapping to the six QPEI domains is a **substantive crosswalk** that must be reviewed by the analyst.

### Final analytic path

```
Data quality
  → Ordinal measurement (polychoric)
  → Factorability + Parallel analysis / MAP / Scree
  → Multi-solution EFA comparison
  → Loading / communality diagnostics
  → Framework crosswalk (review template)
  → Candidate CFA (reflective survey scales only)
  → Item decision log
  → QPEI indicator validation path
  → School aggregation → weighting → sensitivity → final QPEI
```

### Six QPEI dimensions (formative school-level composite)

| Code | Dimension | Weight |
|------|-----------|--------|
| D1 | Teacher competence and pedagogical practice | 20% |
| D2 | Curriculum implementation and assessment | 15% |
| D3 | Learning environment and infrastructure | 15% |
| D4 | Student learning outcomes and FLN | 20% |
| D5 | Governance, management, and community support | 15% |
| D6 | Equity and inclusiveness | 15% |

The QPEI is **not** declared valid merely because an EFA/CFA fits.  
Measurement evidence and the conceptual framework must agree.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================

!pip -q install factor_analyzer pingouin semopy openpyxl statsmodels scikit-learn scipy

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, re, warnings, math, traceback
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.stats as st
from scipy.linalg import eigh
import matplotlib.pyplot as plt
import seaborn as sns

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import pingouin as pg
from semopy import Model, calc_stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")

# ---------- Paths (corrected master) ----------
DATA_PATH = Path("/content/drive/MyDrive/QPEI/QPE_MASTER_Cleaned_IDs_Translated_Analysis.xlsx")
RESULTS_PATH = Path("/content/drive/MyDrive/QPEI/results_round3.json")
OUTPUT_DIR = Path("/content/drive/MyDrive/QPEI/qpei_validation_round3")

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
DECISION_DIR = OUTPUT_DIR / "decisions"

for p in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, LOG_DIR, DECISION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("QPEI VALIDATION ROUND 3")
print("=" * 100)
print("Data     :", DATA_PATH)
print("Exists   :", DATA_PATH.exists())
print("Results  :", RESULTS_PATH)
print("Output   :", OUTPUT_DIR)
print("Started  :", datetime.now().isoformat(timespec="seconds"))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Master workbook not found: {DATA_PATH}")


# 1. Load workbook and exact item inventory

Only items whose column names match the strict pattern `tq|pq|sq|co|se` + digits are treated as psychometric items.


In [ ]:
# ============================================================
# 1. LOAD WORKBOOK + EXACT ITEM INVENTORY
# ============================================================

xl = pd.ExcelFile(DATA_PATH)
print("Sheets:")
for s in xl.sheet_names:
    print(" •", s)

sheets = {s: pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names}

INSTRUMENTS = {
    "Teacher":     {"sheet": "Teacher_Survey",         "prefix": "tq"},
    "Parent":      {"sheet": "Parent_Survey",          "prefix": "pq"},
    "Student":     {"sheet": "Student_Questionnaire",  "prefix": "sq"},
    "Observation": {"sheet": "Classroom_Observation",  "prefix": "co"},
    "Environment": {"sheet": "School_Environment",     "prefix": "se"},
}

ITEMS = {}
for label, spec in INSTRUMENTS.items():
    df = sheets[spec["sheet"]]
    prefix = spec["prefix"]
    cols = [
        c for c in df.columns
        if re.fullmatch(fr"{prefix}\d+", str(c).strip().lower())
    ]
    cols = sorted(cols, key=lambda x: int(re.search(r"\d+", str(x)).group()))
    ITEMS[label] = cols
    print(f"{label:12s}: {len(cols):2d} items → {cols}")

TQ_ITEMS = ITEMS["Teacher"]
PQ_ITEMS = ITEMS["Parent"]
SQ_ITEMS = ITEMS["Student"]

assert len(TQ_ITEMS) == 30, f"Expected 30 teacher items, got {len(TQ_ITEMS)}"
assert len(PQ_ITEMS) == 20, f"Expected 20 parent items, got {len(PQ_ITEMS)}"
assert len(SQ_ITEMS) == 18, f"Expected 18 student items, got {len(SQ_ITEMS)}"

inventory = pd.DataFrame([
    {
        "instrument": k,
        "sheet": v["sheet"],
        "n_items": len(ITEMS[k]),
        "items": ", ".join(ITEMS[k]),
    }
    for k, v in INSTRUMENTS.items()
])

display(inventory)
inventory.to_csv(TABLE_DIR / "R3_Table_01_Item_Inventory.csv", index=False)


# 2. Response cleaning and missingness

The cleaned master uses:

- **1–5** = substantive ordinal response
- **88** = Not Applicable
- **99** = Missing / Unclear

For psychometric analysis, 88 and 99 are set to missing (`NaN`).  
The original workbook is never altered; only analysis copies are modified.


In [ ]:
# ============================================================
# 2. CLEAN ANALYSIS COPIES + MISSINGNESS AUDIT
# ============================================================

analysis_sheets = {}

for name, df in sheets.items():
    x = df.copy()
    for col in x.columns:
        if re.fullmatch(r"(tq|pq|sq|co|se)\d+", str(col).strip().lower()):
            x[col] = pd.to_numeric(x[col], errors="coerce")
            x.loc[x[col].isin([88, 99]), col] = np.nan
    analysis_sheets[name] = x

missing_rows = []
for label in ["Teacher", "Parent", "Student"]:
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    for item in ITEMS[label]:
        s = df[item]
        n_total = len(s)
        n_valid = int(s.notna().sum())
        n_missing = n_total - n_valid
        missing_rows.append({
            "instrument": label,
            "item": item,
            "N_total": n_total,
            "N_valid": n_valid,
            "N_missing": n_missing,
            "missing_pct": round(100.0 * n_missing / n_total, 2) if n_total else np.nan,
            "mean": float(s.mean()) if n_valid else np.nan,
            "sd": float(s.std(ddof=1)) if n_valid > 1 else np.nan,
            "min": float(s.min()) if n_valid else np.nan,
            "max": float(s.max()) if n_valid else np.nan,
        })

missing_table = pd.DataFrame(missing_rows)
display(missing_table)
missing_table.to_csv(TABLE_DIR / "R3_Table_02_Missingness_Item_Distribution.csv", index=False)

print("\nHigh missingness (≥10%):")
hi = missing_table[missing_table["missing_pct"] >= 10]
print(hi[["instrument", "item", "missing_pct"]].to_string(index=False) if len(hi) else "  None")


In [ ]:
# ============================================================
# 2A. RESPONSE DISTRIBUTIONS (1–5)
# ============================================================

dist_rows = []
for label in ["Teacher", "Parent", "Student"]:
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    for item in ITEMS[label]:
        s = df[item].dropna()
        vc = s.value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0)
        dist_rows.append({
            "instrument": label,
            "item": item,
            "n_valid": int(len(s)),
            "pct_1": round(100 * vc.get(1, 0), 1),
            "pct_2": round(100 * vc.get(2, 0), 1),
            "pct_3": round(100 * vc.get(3, 0), 1),
            "pct_4": round(100 * vc.get(4, 0), 1),
            "pct_5": round(100 * vc.get(5, 0), 1),
            "mean": round(float(s.mean()), 3) if len(s) else np.nan,
            "skew": round(float(s.skew()), 3) if len(s) > 2 else np.nan,
        })

distribution_table = pd.DataFrame(dist_rows)
display(distribution_table.head(20))
distribution_table.to_csv(TABLE_DIR / "R3_Table_03_Response_Distributions.csv", index=False)


# 3. Reverse-coding audit

Reverse coding must be based on **item meaning**, not on statistical output alone.  
Only items that are *explicitly* confirmed as reverse-keyed in the codebook / wording are reversed.  
If no confirmed reverse items exist, the map remains empty and is logged as such.


In [ ]:
# ============================================================
# 3. EXPLICIT REVERSE-CODE MAP
# ============================================================
# Populate ONLY after substantive review of item wording.
# Format: {"instrument": ["item1", "item2", ...]}

CONFIRMED_REVERSE = {
    "Teacher": [],
    "Parent":  [],
    "Student": [],
}

# Optional: attempt to surface candidate reverse wording from Codebook sheet
codebook = sheets.get("Codebook")
reverse_audit_rows = []

for label in ["Teacher", "Parent", "Student"]:
    for item in ITEMS[label]:
        is_rev = item in CONFIRMED_REVERSE.get(label, [])
        reverse_audit_rows.append({
            "instrument": label,
            "item": item,
            "confirmed_reverse": is_rev,
            "note": "Explicitly listed in CONFIRMED_REVERSE" if is_rev else "Not reversed",
        })

reverse_audit = pd.DataFrame(reverse_audit_rows)
display(reverse_audit[reverse_audit["confirmed_reverse"]])
print(f"Total confirmed reverse items: {reverse_audit['confirmed_reverse'].sum()}")
reverse_audit.to_csv(TABLE_DIR / "R3_Table_05_Reverse_Code_Audit.csv", index=False)

# Apply confirmed reverses (1↔5, 2↔4; 3 stays)
def reverse_likert(series):
    mapping = {1: 5, 2: 4, 3: 3, 4: 2, 5: 1}
    return series.map(mapping)

for label, items in CONFIRMED_REVERSE.items():
    if not items:
        continue
    sheet_name = INSTRUMENTS[label]["sheet"]
    df = analysis_sheets[sheet_name]
    for item in items:
        if item in df.columns:
            df[item] = reverse_likert(df[item])
            print(f"Reversed: {label} / {item}")
    analysis_sheets[sheet_name] = df


# 4. Ordinal association matrix (polychoric)

Because items are 5-category ordinal, the preferred association matrix is **polychoric**.  
When a pairwise polychoric estimate fails (empty cells, extreme margins), the pair falls back to **Spearman ρ**, and the fallback is logged.

Before EFA the matrix is checked for **positive-definiteness**.  
If the smallest eigenvalue is ≤ 0, a minimal ridge (`ε·I`) is added and documented.


In [ ]:
# ============================================================
# 4. POLYCHORIC CORRELATION IMPLEMENTATION
# ============================================================

def _pairwise_polychoric(x, y):
    """Estimate polychoric correlation for two ordinal series.
    Returns (rho, method) where method is 'polychoric' or 'spearman_fallback'.
    """
    mask = x.notna() & y.notna()
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    if len(x) < 10:
        return np.nan, "insufficient_n"

    # Try pingouin's polychoric if available; otherwise use a simple 2-step approach
    try:
        # pingouin.polychoric returns (rho, thresholds)
        rho, _ = pg.polychoric(x, y)
        if np.isfinite(rho):
            return float(np.clip(rho, -0.999, 0.999)), "polychoric"
    except Exception:
        pass

    # Fallback: Spearman
    try:
        rho, _ = st.spearmanr(x, y)
        if np.isfinite(rho):
            return float(np.clip(rho, -0.999, 0.999)), "spearman_fallback"
    except Exception:
        pass

    return np.nan, "failed"


def build_ordinal_matrix(df, items, label=""):
    """Build symmetric ordinal correlation matrix + audit log."""
    n = len(items)
    R = np.eye(n)
    audit = []

    for i in range(n):
        for j in range(i + 1, n):
            rho, method = _pairwise_polychoric(df[items[i]], df[items[j]])
            if not np.isfinite(rho):
                rho = 0.0
                method = method + "_set0"
            R[i, j] = R[j, i] = rho
            audit.append({
                "instrument": label,
                "item_i": items[i],
                "item_j": items[j],
                "rho": rho,
                "method": method,
            })

    R_df = pd.DataFrame(R, index=items, columns=items)
    audit_df = pd.DataFrame(audit)
    return R_df, audit_df


def ensure_positive_definite(R_df, eps=1e-6):
    """Return PD matrix and diagnostics."""
    R = R_df.values.copy()
    items = list(R_df.index)
    evals = eigh(R, eigvals_only=True)
    min_ev = float(evals.min())
    repaired = False
    ridge = 0.0

    if min_ev <= 0:
        # Minimal ridge to push smallest eigenvalue just above zero
        ridge = abs(min_ev) + eps
        R = R + np.eye(R.shape[0]) * ridge
        repaired = True
        evals = eigh(R, eigvals_only=True)
        min_ev = float(evals.min())

    # Re-standardise diagonal to 1 (ridge can inflate slightly)
    d = np.sqrt(np.diag(R))
    R = R / np.outer(d, d)
    np.fill_diagonal(R, 1.0)

    out = pd.DataFrame(R, index=items, columns=items)
    info = {
        "min_eigenvalue_before": float(eigh(R_df.values, eigvals_only=True).min()),
        "min_eigenvalue_after": min_ev,
        "ridge_applied": ridge,
        "repaired": repaired,
        "is_pd": min_ev > 0,
    }
    return out, info


In [ ]:
# ============================================================
# 4A. BUILD ORDINAL CORRELATION MATRICES + PD CHECK
# ============================================================

ordinal_corr = {}
ordinal_audit = {}
pd_info = {}

for label in ["Teacher", "Parent", "Student"]:
    print("\n" + "=" * 90)
    print(f"Building ordinal matrix: {label}")
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    items = ITEMS[label]

    R_raw, audit = build_ordinal_matrix(df, items, label=label)
    R_pd, info = ensure_positive_definite(R_raw)

    ordinal_corr[label] = R_pd
    ordinal_audit[label] = audit
    pd_info[label] = info

    print(f"  Pairs: {len(audit)}")
    print(f"  Methods:\n{audit['method'].value_counts().to_string()}")
    print(f"  PD info: {info}")

    R_pd.to_csv(TABLE_DIR / f"R3_Ordinal_Correlation_{label}.csv")
    audit.to_csv(TABLE_DIR / f"R3_Ordinal_Correlation_Audit_{label}.csv", index=False)

pd_summary = pd.DataFrame(pd_info).T
display(pd_summary)
pd_summary.to_csv(TABLE_DIR / "R3_Table_04_Positive_Definiteness.csv")


In [ ]:
# ============================================================
# 4B. ORDINAL KMO + BARTLETT
# ============================================================

factorability_rows = []

for label in ["Teacher", "Parent", "Student"]:
    R = ordinal_corr[label].values
    # KMO / Bartlett expect a correlation matrix; feed the ordinal matrix
    try:
        kmo_all, kmo_model = calculate_kmo(R)
        chi2, p = calculate_bartlett_sphericity(R)
    except Exception as e:
        kmo_model, chi2, p = np.nan, np.nan, np.nan
        print(f"{label}: factorability failed → {e}")

    factorability_rows.append({
        "instrument": label,
        "n_items": R.shape[0],
        "kmo": float(kmo_model) if np.isfinite(kmo_model) else np.nan,
        "bartlett_chi2": float(chi2) if np.isfinite(chi2) else np.nan,
        "bartlett_p": float(p) if np.isfinite(p) else np.nan,
        "factorable": bool(np.isfinite(kmo_model) and kmo_model >= 0.50 and np.isfinite(p) and p < 0.05),
    })

ordinal_factorability = pd.DataFrame(factorability_rows)
display(ordinal_factorability)
ordinal_factorability.to_csv(TABLE_DIR / "R3_Table_06_Ordinal_Factorability.csv", index=False)


# 5. Parallel analysis, scree, and MAP on the ordinal structure

Factor retention is informed by three sources of evidence:

1. **Parallel analysis** (mean and 95th-percentile random eigenvalues)
2. **Scree plot** inspection
3. **Velicer’s MAP** (minimum average partial) when computable

No single rule is treated as definitive; the subsequent multi-solution EFA compares k−1, k, and k+1.


In [ ]:
# ============================================================
# 5. PARALLEL ANALYSIS + MAP
# ============================================================

def parallel_analysis_corr(R, n_obs, n_iter=500, seed=42):
    """Parallel analysis on a correlation matrix."""
    rng = np.random.default_rng(seed)
    p = R.shape[0]
    real_ev = np.sort(eigh(R, eigvals_only=True))[::-1]

    rand_evs = np.zeros((n_iter, p))
    for i in range(n_iter):
        # Random normal data → correlation eigenvalues
        X = rng.standard_normal((n_obs, p))
        Rc = np.corrcoef(X, rowvar=False)
        rand_evs[i] = np.sort(eigh(Rc, eigvals_only=True))[::-1]

    mean_rand = rand_evs.mean(axis=0)
    p95_rand = np.percentile(rand_evs, 95, axis=0)

    suggested_mean = int(np.sum(real_ev > mean_rand))
    suggested_95 = int(np.sum(real_ev > p95_rand))

    table = pd.DataFrame({
        "factor": np.arange(1, p + 1),
        "real_eigenvalue": real_ev,
        "mean_random": mean_rand,
        "p95_random": p95_rand,
        "gt_mean": real_ev > mean_rand,
        "gt_p95": real_ev > p95_rand,
    })
    return {
        "table": table,
        "suggested_mean": max(suggested_mean, 1),
        "suggested_95": max(suggested_95, 1),
        "real_ev": real_ev,
        "mean_rand": mean_rand,
        "p95_rand": p95_rand,
    }


def velicer_map(R):
    """Velicer's Minimum Average Partial (MAP) on correlation matrix.
    Returns (suggested_k, map_values array).
    """
    p = R.shape[0]
    # Eigen-decomposition
    evals, evecs = eigh(R)
    idx = np.argsort(evals)[::-1]
    evals, evecs = evals[idx], evecs[:, idx]

    map_vals = []
    for k in range(0, p - 1):
        if k == 0:
            # partial correlations among original variables = off-diag of R
            C = R.copy()
        else:
            # residual covariance after k principal components
            Lk = evecs[:, :k] * np.sqrt(evals[:k])
            residual = R - Lk @ Lk.T
            # convert residual cov to correlation
            d = np.sqrt(np.diag(residual))
            d[d < 1e-12] = 1e-12
            C = residual / np.outer(d, d)
            np.fill_diagonal(C, 1.0)

        # average squared partial correlation (off-diagonal)
        off = C[np.triu_indices_from(C, k=1)]
        map_vals.append(float(np.mean(off ** 2)))

    map_vals = np.array(map_vals)
    suggested = int(np.argmin(map_vals))  # k that minimises MAP (0-based → number of factors)
    # MAP index 0 = 0 factors; we want at least 1 for practical use
    return max(suggested, 1), map_vals


parallel3 = {}
map3 = {}

for label in ["Teacher", "Parent", "Student"]:
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    n_obs = int(df[ITEMS[label]].notna().all(axis=1).sum())
    if n_obs < 30:
        n_obs = int(df[ITEMS[label]].notna().any(axis=1).sum())

    R = ordinal_corr[label].values
    pa = parallel_analysis_corr(R, n_obs=max(n_obs, 50), n_iter=400)
    map_k, map_vals = velicer_map(R)

    parallel3[label] = pa
    map3[label] = {"suggested": map_k, "values": map_vals}

    print("\n" + "=" * 90)
    print(f"{label}")
    print(f"  Parallel (mean) : {pa['suggested_mean']}")
    print(f"  Parallel (95%)  : {pa['suggested_95']}")
    print(f"  MAP             : {map_k}")
    display(pa["table"].head(8))

    pa["table"].to_csv(TABLE_DIR / f"R3_Table_07_Parallel_Analysis_{label}.csv", index=False)

    # Scree + PA figure
    fig, ax = plt.subplots(figsize=(8, 5))
    k = np.arange(1, len(pa["real_ev"]) + 1)
    ax.plot(k, pa["real_ev"], "o-", label="Real eigenvalues", color="#1f77b4")
    ax.plot(k, pa["mean_rand"], "s--", label="Mean random", color="#ff7f0e", alpha=0.8)
    ax.plot(k, pa["p95_rand"], "^--", label="95th pct random", color="#d62728", alpha=0.8)
    ax.axhline(1.0, color="grey", ls=":", lw=1)
    ax.set_xlabel("Factor")
    ax.set_ylabel("Eigenvalue")
    ax.set_title(f"Parallel Analysis — {label}")
    ax.legend()
    ax.set_xticks(k[:: max(1, len(k) // 10)])
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"R3_Figure_01_Parallel_{label}.png", dpi=150)
    plt.show()


# 6. Multi-solution ordinal EFA

Extraction: **Principal Axis Factoring** on the ordinal (PD-repaired) correlation matrix.  
Rotation: **Oblimin** (oblique) because latent dimensions of school quality are expected to correlate.

For each instrument we fit **three** solutions:

- k − 1  (where k = parallel-analysis mean suggestion)
- k
- k + 1

and retain diagnostics for all three.  The preferred solution is chosen later by the analyst on the basis of:

- interpretability
- loading clarity
- communalities
- factor correlations
- alignment with the substantive crosswalk

**No automatic “Factor i = Domain i” assignment is performed.**


In [ ]:
# ============================================================
# 6. MULTI-SOLUTION ORDINAL EFA
# ============================================================

def run_efa_on_corr(R_df, n_factors, rotation="oblimin"):
    """PAF + oblique rotation on a correlation matrix."""
    fa = FactorAnalyzer(
        n_factors=n_factors,
        method="principal",          # PAF-style on correlation
        rotation=rotation,
        is_corr_matrix=True,
    )
    fa.fit(R_df.values)

    loadings = pd.DataFrame(
        fa.loadings_,
        index=R_df.index,
        columns=[f"F{i+1}" for i in range(n_factors)],
    )
    communalities = pd.Series(
        fa.get_communalities(),
        index=R_df.index,
        name="communality",
    )

    # Factor correlation matrix (phi) when oblique
    try:
        phi = fa.phi_
        if phi is not None:
            factor_corr = pd.DataFrame(
                phi,
                index=loadings.columns,
                columns=loadings.columns,
            )
        else:
            factor_corr = pd.DataFrame(
                np.eye(n_factors),
                index=loadings.columns,
                columns=loadings.columns,
            )
    except Exception:
        factor_corr = pd.DataFrame(
            np.eye(n_factors),
            index=loadings.columns,
            columns=loadings.columns,
        )

    return {
        "n_factors": n_factors,
        "loadings": loadings,
        "communalities": communalities,
        "factor_correlations": factor_corr,
        "fa": fa,
    }


def loading_clarity_score(loadings, primary_cut=0.40, gap_cut=0.10):
    """Proportion of items with clean primary loading."""
    L = loadings.abs()
    primary = L.max(axis=1)
    if L.shape[1] == 1:
        secondary = pd.Series(0.0, index=L.index)
    else:
        secondary = L.apply(lambda row: row.nlargest(2).iloc[-1], axis=1)
    clean = (primary >= primary_cut) & ((primary - secondary) >= gap_cut)
    return float(clean.mean()), int(clean.sum()), len(clean)


ordinal_efa_solutions = {}   # label → {k: result}
ordinal_efa = {}             # preferred (middle) solution for downstream use
efa_comparison = []

for label in ["Teacher", "Parent", "Student"]:
    R_df = ordinal_corr[label]
    k_base = parallel3[label]["suggested_mean"]
    candidates = sorted(set([
        max(1, k_base - 1),
        k_base,
        min(R_df.shape[0] // 2, k_base + 1),
    ]))

    ordinal_efa_solutions[label] = {}
    print("\n" + "=" * 100)
    print(f"EFA multi-solution: {label}  |  candidates = {candidates}")

    for k in candidates:
        try:
            res = run_efa_on_corr(R_df, n_factors=k, rotation="oblimin")
            clarity, n_clean, n_tot = loading_clarity_score(res["loadings"])
            mean_h2 = float(res["communalities"].mean())
            low_h2 = int((res["communalities"] < 0.20).sum())

            ordinal_efa_solutions[label][k] = res
            efa_comparison.append({
                "instrument": label,
                "n_factors": k,
                "clarity_pct": round(100 * clarity, 1),
                "n_clean_items": n_clean,
                "n_items": n_tot,
                "mean_communality": round(mean_h2, 3),
                "n_low_communality_<0.20": low_h2,
                "is_base_pa": k == k_base,
            })

            print(f"  k={k}: clarity={100*clarity:.1f}%  mean_h2={mean_h2:.3f}  low_h2={low_h2}")

            # Save loadings / communalities / factor correlations for every solution
            res["loadings"].to_csv(
                TABLE_DIR / f"R3_Table_08_EFA_Loadings_{label}_k{k}.csv"
            )
            res["communalities"].to_frame().to_csv(
                TABLE_DIR / f"R3_Table_09_EFA_Communalities_{label}_k{k}.csv"
            )
            res["factor_correlations"].to_csv(
                TABLE_DIR / f"R3_Table_10_EFA_FactorCorr_{label}_k{k}.csv"
            )

        except Exception as e:
            print(f"  k={k} FAILED: {e}")
            traceback.print_exc()

    # Preferred = base PA suggestion if it succeeded, else first available
    if k_base in ordinal_efa_solutions[label]:
        ordinal_efa[label] = ordinal_efa_solutions[label][k_base]
    elif ordinal_efa_solutions[label]:
        first_k = sorted(ordinal_efa_solutions[label].keys())[0]
        ordinal_efa[label] = ordinal_efa_solutions[label][first_k]
    else:
        print(f"WARNING: No EFA solution for {label}")

efa_comparison_df = pd.DataFrame(efa_comparison)
display(efa_comparison_df)
efa_comparison_df.to_csv(TABLE_DIR / "R3_Table_08b_EFA_Solution_Comparison.csv", index=False)


In [ ]:
# ============================================================
# 6A. LOADING / COMMUNALITY DIAGNOSTICS (preferred solution)
# ============================================================

PRIMARY_CUT = 0.40
SECONDARY_CUT = 0.30
GAP_CUT = 0.15
COMM_CUT = 0.20

loading_flags = []

for label in ["Teacher", "Parent", "Student"]:
    if label not in ordinal_efa:
        continue
    L = ordinal_efa[label]["loadings"]
    H = ordinal_efa[label]["communalities"]

    for item in L.index:
        abs_row = L.loc[item].abs().sort_values(ascending=False)
        primary = float(abs_row.iloc[0])
        primary_f = abs_row.index[0]
        secondary = float(abs_row.iloc[1]) if len(abs_row) > 1 else 0.0
        secondary_f = abs_row.index[1] if len(abs_row) > 1 else None
        h2 = float(H.loc[item])

        flag_weak = primary < PRIMARY_CUT
        flag_cross = (
            primary >= PRIMARY_CUT
            and secondary >= SECONDARY_CUT
            and (primary - secondary) < GAP_CUT
        )
        flag_low_h2 = h2 < COMM_CUT

        loading_flags.append({
            "instrument": label,
            "item": item,
            "primary_factor": primary_f,
            "primary_loading": round(primary, 3),
            "secondary_factor": secondary_f,
            "secondary_loading": round(secondary, 3),
            "loading_gap": round(primary - secondary, 3),
            "communality": round(h2, 3),
            "flag_weak_loading": flag_weak,
            "flag_cross_loading": flag_cross,
            "flag_low_communality": flag_low_h2,
        })

loading_flags = pd.DataFrame(loading_flags)
display(loading_flags)
loading_flags.to_csv(TABLE_DIR / "R3_Table_11_EFA_Loading_Flags.csv", index=False)

print("\nFlag summary:")
print(loading_flags.groupby("instrument")[
    ["flag_weak_loading", "flag_cross_loading", "flag_low_communality"]
].sum())


In [ ]:
# ============================================================
# 6B. EFA LOADING HEATMAPS (preferred solution)
# ============================================================

for label in ["Teacher", "Parent", "Student"]:
    if label not in ordinal_efa:
        continue
    L = ordinal_efa[label]["loadings"]
    fig, ax = plt.subplots(figsize=(max(6, L.shape[1] * 1.2), max(6, L.shape[0] * 0.28)))
    sns.heatmap(
        L, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        vmin=-1, vmax=1, ax=ax, linewidths=0.3,
    )
    ax.set_title(f"Ordinal EFA Loadings — {label} (k={ordinal_efa[label]['n_factors']})")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"R3_Figure_02_EFA_Loadings_{label}.png", dpi=150)
    plt.show()


# 7. QPEI framework and empirical crosswalk template

The six domains and their indicator families are **theory-defined**.  
Empirical factors (F1, F2, …) are **not** automatically mapped onto domains.

The crosswalk table is a **review template**: the analyst fills in the suggested QPEI domain after inspecting item wording and loadings.


In [ ]:
# ============================================================
# 7. QPEI FRAMEWORK (35 indicators / 6 domains)
# ============================================================

QPEI_FRAMEWORK = {
    "D1": {
        "name": "Teacher competence and pedagogical practice",
        "weight": 0.20,
        "indicators": [
            "Subject knowledge",
            "Pedagogical skill",
            "Lesson planning",
            "Classroom management",
            "Use of teaching aids",
            "Feedback to students",
            "Professional development",
        ],
    },
    "D2": {
        "name": "Curriculum implementation and assessment",
        "weight": 0.15,
        "indicators": [
            "Curriculum coverage",
            "Lesson alignment",
            "Formative assessment",
            "Summative assessment",
            "Use of assessment data",
        ],
    },
    "D3": {
        "name": "Learning environment and infrastructure",
        "weight": 0.15,
        "indicators": [
            "Classroom physical condition",
            "Seating / space",
            "Teaching-learning materials",
            "Sanitation / water",
            "Safety",
            "Digital / library resources",
        ],
    },
    "D4": {
        "name": "Student learning outcomes and FLN",
        "weight": 0.20,
        "indicators": [
            "Reading",
            "Writing",
            "Numeracy",
            "Participation",
            "Confidence",
            "Aspiration",
        ],
    },
    "D5": {
        "name": "Governance, management, and community support",
        "weight": 0.15,
        "indicators": [
            "Leadership",
            "Supervision",
            "Monitoring",
            "Parent engagement",
            "Data use",
        ],
    },
    "D6": {
        "name": "Equity and inclusiveness",
        "weight": 0.15,
        "indicators": [
            "Equal participation",
            "Slow learner support",
            "Gender sensitivity",
            "Support for disadvantaged learners",
        ],
    },
}

QPEI_WEIGHTS = {k: v["weight"] for k, v in QPEI_FRAMEWORK.items()}
assert abs(sum(QPEI_WEIGHTS.values()) - 1.0) < 1e-9

framework_rows = []
for d, spec in QPEI_FRAMEWORK.items():
    for ind in spec["indicators"]:
        framework_rows.append({
            "domain": d,
            "domain_name": spec["name"],
            "weight": spec["weight"],
            "indicator": ind,
        })

framework_df = pd.DataFrame(framework_rows)
display(framework_df)
framework_df.to_csv(TABLE_DIR / "R3_Table_12_QPEI_Framework.csv", index=False)

pd.Series(QPEI_WEIGHTS, name="weight").to_csv(
    TABLE_DIR / "R3_Table_18_QPEI_Weights.csv"
)


In [ ]:
# ============================================================
# 7A. EMPIRICAL ↔ FRAMEWORK CROSSWALK TEMPLATE
# ============================================================
# This is a REVIEW template.  Columns left for the analyst:
#   suggested_qpei_domain, suggested_indicator, analyst_note, decision

crosswalk_rows = []

for label in ["Teacher", "Parent", "Student"]:
    if label not in ordinal_efa:
        continue
    L = ordinal_efa[label]["loadings"]
    H = ordinal_efa[label]["communalities"]
    flags = loading_flags[loading_flags["instrument"] == label].set_index("item")

    for item in L.index:
        abs_row = L.loc[item].abs().sort_values(ascending=False)
        primary_f = abs_row.index[0]
        primary_l = float(abs_row.iloc[0])

        crosswalk_rows.append({
            "instrument": label,
            "item": item,
            "empirical_factor": primary_f,
            "primary_loading": round(primary_l, 3),
            "communality": round(float(H.loc[item]), 3),
            "flag_weak": bool(flags.loc[item, "flag_weak_loading"]) if item in flags.index else False,
            "flag_cross": bool(flags.loc[item, "flag_cross_loading"]) if item in flags.index else False,
            "flag_low_h2": bool(flags.loc[item, "flag_low_communality"]) if item in flags.index else False,
            # --- analyst fill-in columns ---
            "suggested_qpei_domain": "",
            "suggested_indicator": "",
            "analyst_note": "",
            "decision": "REVIEW",   # RETAIN / REVIEW / DROP
        })

crosswalk = pd.DataFrame(crosswalk_rows)
display(crosswalk)
crosswalk.to_csv(
    TABLE_DIR / "R3_Table_13_Empirical_QPEI_Crosswalk_REVIEW.csv",
    index=False,
)
print("\nCrosswalk status: REQUIRES_SUBSTANTIVE_REVIEW")
print("No automatic Factor → Domain assignment has been performed.")


# 8. Reliability of empirical factor indicator sets

Cronbach’s α and (when feasible) ordinal α are reported for the indicator sets of each empirical factor in the preferred solution.  
These are **reflective survey-scale** reliability estimates, not claims about the formative QPEI composite.


In [ ]:
# ============================================================
# 8. RELIABILITY OF EFA FACTOR INDICATOR SETS
# ============================================================

reliability_rows = []

for label in ["Teacher", "Parent", "Student"]:
    if label not in ordinal_efa:
        continue
    L = ordinal_efa[label]["loadings"]
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]

    for factor in L.columns:
        abs_primary = L[factor].abs()
        if L.shape[1] > 1:
            abs_other = L.drop(columns=[factor]).abs().max(axis=1)
        else:
            abs_other = pd.Series(0.0, index=L.index)

        selected = L.index[
            (abs_primary >= PRIMARY_CUT) & ((abs_primary - abs_other) >= GAP_CUT)
        ].tolist()

        if len(selected) < 2:
            reliability_rows.append({
                "instrument": label,
                "factor": factor,
                "n_items": len(selected),
                "items": ",".join(selected),
                "cronbach_alpha": np.nan,
                "note": "<2 clean indicators",
            })
            continue

        sub = df[selected].dropna(how="any")
        if len(sub) < 10:
            alpha = np.nan
            note = "insufficient complete cases"
        else:
            try:
                alpha = float(pg.cronbach_alpha(data=sub)[0])
                note = "ok"
            except Exception as e:
                alpha = np.nan
                note = str(e)[:80]

        reliability_rows.append({
            "instrument": label,
            "factor": factor,
            "n_items": len(selected),
            "items": ",".join(selected),
            "cronbach_alpha": round(alpha, 3) if np.isfinite(alpha) else np.nan,
            "note": note,
        })

reliability_r3 = pd.DataFrame(reliability_rows)
display(reliability_r3)
reliability_r3.to_csv(TABLE_DIR / "R3_Table_14_Reliability.csv", index=False)


# 9. Candidate CFA (reflective survey scales only)

CFA is run **only** on the empirical factors that already have ≥ 3 clean indicators from the preferred EFA solution.  
Models are fitted with `semopy` on the raw ordinal scores (listwise).  
Fit indices are reported for transparency; they do **not** validate the formative QPEI composite.

**Do not fit** `QPEI =~ D1 + D2 + D3 + D4 + D5 + D6`.


In [ ]:
# ============================================================
# 9. GENERATE CANDIDATE CFA SYNTAX
# ============================================================

CFA_MODELS = {}

for label in ["Teacher", "Parent", "Student"]:
    if label not in ordinal_efa:
        CFA_MODELS[label] = ""
        continue

    L = ordinal_efa[label]["loadings"]
    lines = []

    for factor in L.columns:
        abs_primary = L[factor].abs()
        if L.shape[1] > 1:
            abs_other = L.drop(columns=[factor]).abs().max(axis=1)
        else:
            abs_other = pd.Series(0.0, index=L.index)

        selected = L.index[
            (abs_primary >= PRIMARY_CUT) & ((abs_primary - abs_other) >= GAP_CUT)
        ].tolist()

        if len(selected) >= 3:
            lines.append(f"{factor} =~ " + " + ".join(selected))

    # Correlate factors if ≥ 2
    factors_used = [ln.split(" =~ ")[0] for ln in lines]
    if len(factors_used) >= 2:
        for i in range(len(factors_used)):
            for j in range(i + 1, len(factors_used)):
                lines.append(f"{factors_used[i]} ~~ {factors_used[j]}")

    CFA_MODELS[label] = "\n".join(lines)

    print("\n" + "=" * 90)
    print(label)
    print(CFA_MODELS[label] if CFA_MODELS[label] else "No candidate factor met the ≥3-indicator rule.")

    (LOG_DIR / f"R3_CFA_Candidate_{label}.txt").write_text(
        CFA_MODELS[label] or "(empty)", encoding="utf-8"
    )


In [ ]:
# ============================================================
# 9A. FIT CANDIDATE CFA
# ============================================================

cfa_r3 = {}

for label in ["Teacher", "Parent", "Student"]:
    syntax = CFA_MODELS.get(label, "")
    if not syntax.strip():
        print(f"{label}: skipped (no model)")
        continue

    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    # Collect all items appearing in syntax
    items_in_model = sorted(set(re.findall(r"\b[tps]q\d+\b", syntax)))
    data = df[items_in_model].dropna(how="any")

    print("\n" + "=" * 90)
    print(f"{label}: N complete = {len(data)}  |  items = {items_in_model}")

    if len(data) < 50:
        print("  Insufficient N for stable CFA; recorded as failed.")
        cfa_r3[label] = {"status": "insufficient_n", "n": len(data)}
        continue

    try:
        model = Model(syntax)
        model.fit(data)
        stats = calc_stats(model)
        # stats is a DataFrame-like object
        stats_df = stats.T if hasattr(stats, "T") else pd.DataFrame(stats)
        stats_df.to_csv(TABLE_DIR / f"R3_Table_15_CFA_Fit_{label}.csv")

        # Standardised loadings
        try:
            est = model.inspect(std_est=True)
            est.to_csv(TABLE_DIR / f"R3_Table_16_CFA_Standardized_{label}.csv", index=False)
        except Exception:
            est = None

        cfa_r3[label] = {
            "status": "fitted",
            "n": len(data),
            "syntax": syntax,
            "stats": stats_df.to_dict() if hasattr(stats_df, "to_dict") else str(stats_df),
        }
        print("  Fit stats:")
        display(stats_df)

    except Exception as e:
        print(f"  CFA failed: {e}")
        cfa_r3[label] = {"status": "failed", "error": str(e)[:200]}


# 10. Machine-readable item decision log

Every item receives an explicit decision code and rationale derived from the quantitative flags.  
Final retain / drop decisions still require substantive review of the crosswalk; the automated layer only supplies transparent defaults.

| Code | Meaning |
|------|---------|
| RETAIN_CANDIDATE | Clean primary loading, adequate communality, low missingness |
| REVIEW_CROSSLOAD | Cross-loading or small loading gap |
| REVIEW_WEAK | Primary loading below threshold |
| REVIEW_LOW_H2 | Communality below threshold |
| REVIEW_MISSING | Missingness ≥ 10% |
| REVIEW | Multiple or residual concerns |


In [ ]:
# ============================================================
# 10. ITEM DECISION LOG
# ============================================================

decision_rows = []

for label in ["Teacher", "Parent", "Student"]:
    miss = missing_table[missing_table["instrument"] == label].set_index("item")
    flags = loading_flags[loading_flags["instrument"] == label].set_index("item") if label in loading_flags["instrument"].values else pd.DataFrame()

    for item in ITEMS[label]:
        missing_pct = float(miss.loc[item, "missing_pct"]) if item in miss.index else np.nan

        if item in flags.index:
            row = flags.loc[item]
            primary = float(row["primary_loading"])
            secondary = float(row["secondary_loading"])
            gap = float(row["loading_gap"])
            h2 = float(row["communality"])
            primary_f = row["primary_factor"]
            flag_weak = bool(row["flag_weak_loading"])
            flag_cross = bool(row["flag_cross_loading"])
            flag_low_h2 = bool(row["flag_low_communality"])
        else:
            primary = secondary = gap = h2 = np.nan
            primary_f = None
            flag_weak = flag_cross = flag_low_h2 = True

        flag_missing = bool(missing_pct >= 10) if np.isfinite(missing_pct) else False

        reasons = []
        if flag_weak:
            reasons.append("weak_loading")
        if flag_cross:
            reasons.append("cross_loading")
        if flag_low_h2:
            reasons.append("low_communality")
        if flag_missing:
            reasons.append("high_missingness")

        if not reasons:
            decision = "RETAIN_CANDIDATE"
        elif reasons == ["cross_loading"]:
            decision = "REVIEW_CROSSLOAD"
        elif reasons == ["weak_loading"]:
            decision = "REVIEW_WEAK"
        elif reasons == ["low_communality"]:
            decision = "REVIEW_LOW_H2"
        elif reasons == ["high_missingness"]:
            decision = "REVIEW_MISSING"
        else:
            decision = "REVIEW"

        decision_rows.append({
            "instrument": label,
            "item": item,
            "primary_factor": primary_f,
            "primary_loading": primary,
            "secondary_loading": secondary,
            "loading_gap": gap,
            "communality": h2,
            "missing_pct": missing_pct,
            "flag_weak_loading": flag_weak,
            "flag_cross_loading": flag_cross,
            "flag_low_communality": flag_low_h2,
            "flag_high_missingness": flag_missing,
            "decision": decision,
            "reasons": ";".join(reasons) if reasons else "none",
            "analyst_override": "",          # fill after substantive review
            "final_decision": decision,      # copy; override later if needed
            "final_note": "",
        })

decision_log = pd.DataFrame(decision_rows)
display(decision_log)

decision_log.to_csv(TABLE_DIR / "R3_Table_17_Item_Decision_Flags.csv", index=False)
decision_log.to_csv(DECISION_DIR / "R3_Item_Decision_Log.csv", index=False)

print("\nDecision counts:")
print(decision_log.groupby(["instrument", "decision"]).size().unstack(fill_value=0))


# 11. School-level respondent counts

School aggregation for the formative QPEI requires knowing how many respondents contribute per school per instrument.  
These counts feed later weighting / sensitivity analyses; they are not used to alter the psychometric models above.


In [ ]:
# ============================================================
# 11. SCHOOL-LEVEL RESPONDENT COUNTS
# ============================================================

school_counts = {}

for label in ["Teacher", "Parent", "Student"]:
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    # Try common school-id column names
    id_candidates = [c for c in df.columns if re.search(r"school", str(c), re.I)]
    if not id_candidates:
        id_candidates = [c for c in df.columns if re.search(r"^(school_id|sid|sch_id)$", str(c), re.I)]

    if not id_candidates:
        print(f"{label}: no school-id column found; skipping counts")
        school_counts[label] = pd.DataFrame(columns=["school_id", "n_respondents"])
        continue

    sid = id_candidates[0]
    counts = (
        df.groupby(sid)
        .size()
        .reset_index(name="n_respondents")
        .rename(columns={sid: "school_id"})
        .sort_values("n_respondents", ascending=False)
    )
    school_counts[label] = counts
    counts.to_csv(TABLE_DIR / f"R3_School_Respondent_Counts_{label}.csv", index=False)
    print(f"{label}: {len(counts)} schools  |  median n = {counts['n_respondents'].median():.0f}")
    display(counts.head(8))


# 12. Weighting reminder and next-step path

The formative QPEI uses fixed domain weights:

- D1 = 20%  
- D2 = 15%  
- D3 = 15%  
- D4 = 20%  
- D5 = 15%  
- D6 = 15%  

School-level scoring, sensitivity analysis, and final composite construction are **out of scope for this psychometric round**; they belong to the subsequent QPEI construction notebook that consumes the decision log and crosswalk produced here.


In [ ]:
# ============================================================
# 13. ROUND 3 RESULTS JSON
# ============================================================

def json_safe(obj):
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, (float, np.floating)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.DataFrame):
        return {
            "type": "DataFrame",
            "rows": int(obj.shape[0]),
            "columns": [str(c) for c in obj.columns],
            "records": json_safe(obj.replace({np.nan: None}).to_dict(orient="records")),
        }
    if isinstance(obj, pd.Series):
        return json_safe(obj.tolist())
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    if isinstance(obj, np.ndarray):
        return json_safe(obj.tolist())
    return str(obj)


ROUND3_RESULTS = {
    "metadata": {
        "analysis": "QPEI Validation Round 3",
        "data": str(DATA_PATH),
        "generated": datetime.now().isoformat(timespec="seconds"),
        "design_rules": [
            "No automatic Factor → Domain assignment",
            "QPEI remains a formative six-domain composite",
            "CFA applied only to reflective survey-scale candidates",
            "Multi-solution EFA compared (k-1, k, k+1)",
            "Positive-definiteness checked and documented",
        ],
    },
    "sample_sizes": {
        label: int(len(analysis_sheets[INSTRUMENTS[label]["sheet"]]))
        for label in ["Teacher", "Parent", "Student"]
    },
    "item_counts": {label: len(ITEMS[label]) for label in ["Teacher", "Parent", "Student"]},
    "missingness": json_safe(missing_table),
    "response_distributions": json_safe(distribution_table),
    "reverse_code_audit": json_safe(reverse_audit),
    "positive_definiteness": json_safe(pd_summary),
    "ordinal_factorability": json_safe(ordinal_factorability),
    "parallel_analysis": {
        label: {
            "suggested_mean": int(v["suggested_mean"]),
            "suggested_95": int(v["suggested_95"]),
            "map_suggested": int(map3[label]["suggested"]),
        }
        for label, v in parallel3.items()
    },
    "efa_solution_comparison": json_safe(efa_comparison_df),
    "ordinal_efa_preferred": {
        label: {
            "n_factors": int(v["n_factors"]),
            "loadings": json_safe(v["loadings"]),
            "communalities": json_safe(v["communalities"]),
            "factor_correlations": json_safe(v["factor_correlations"]),
        }
        for label, v in ordinal_efa.items()
    },
    "loading_flags": json_safe(loading_flags),
    "reliability": json_safe(reliability_r3),
    "candidate_cfa_models": json_safe(CFA_MODELS),
    "cfa": json_safe(cfa_r3),
    "framework": json_safe(QPEI_FRAMEWORK),
    "weights": json_safe(QPEI_WEIGHTS),
    "school_counts": json_safe(school_counts),
    "decision_log_summary": json_safe(
        decision_log.groupby(["instrument", "decision"]).size().reset_index(name="n")
    ),
    "output_files": [],
    "status": {
        "ordinal_efa_completed": bool(ordinal_efa),
        "multi_solution_efa_completed": bool(ordinal_efa_solutions),
        "cfa_completed": bool(cfa_r3),
        "crosswalk_status": "REQUIRES_SUBSTANTIVE_REVIEW",
        "decision_log_status": "AUTO_FLAGS_GENERATED_PENDING_ANALYST_OVERRIDE",
        "qpei_status": "NOT_YET_SCORED",
        "final_validation_status": "PRELIMINARY_ROUND_3",
    },
}

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        ROUND3_RESULTS["output_files"].append({
            "file": str(p.relative_to(OUTPUT_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 2),
        })

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(json_safe(ROUND3_RESULTS), f, ensure_ascii=False, indent=2, allow_nan=False)

print("=" * 100)
print("ROUND 3 RESULTS JSON SAVED")
print("=" * 100)
print(RESULTS_PATH)
print("Status:", ROUND3_RESULTS["status"])


In [ ]:
# ============================================================
# 14. FINAL OUTPUT MANIFEST
# ============================================================

manifest = []
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "relative_path": str(p.relative_to(OUTPUT_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 2),
            "type": p.suffix,
        })

manifest = pd.DataFrame(manifest)
display(manifest)
manifest.to_csv(OUTPUT_DIR / "R3_OUTPUT_MANIFEST.csv", index=False)

print("\n" + "=" * 100)
print("ROUND 3 COMPLETE")
print("=" * 100)
print("Output directory :", OUTPUT_DIR)
print("Results JSON     :", RESULTS_PATH)
print("Decision log     :", DECISION_DIR / "R3_Item_Decision_Log.csv")
print("Files generated  :", len(manifest))
print("\nNext steps:")
print("  1. Substantive review of R3_Table_13_Empirical_QPEI_Crosswalk_REVIEW.csv")
print("  2. Analyst overrides in R3_Item_Decision_Log.csv")
print("  3. Proceed to school-level aggregation / weighting / sensitivity notebook")


# Interpretation rule for the manuscript

Do **not** write:

> “The QPEI was validated because CFA fit was acceptable.”

The final validity argument must integrate:

- content validity (framework + indicator definitions)
- response-process evidence (translation, reverse-coding, missingness)
- internal structure of respondent instruments (ordinal EFA + multi-solution comparison + CFA of reflective scales)
- convergent / triangulated evidence across Teacher, Parent, Student, Observation, Environment
- school-level aggregation evidence
- sensitivity of the composite to weighting and indicator choices
- theoretical coherence of the six domains

The QPEI is a **theory-informed formative composite**.  
The six dimensions must **not** be collapsed into a single reflective latent factor simply to obtain a conventional CFA fit index.

Empirical factors remain labelled F1, F2, … until a human reviewer completes the crosswalk.
